<a href="https://colab.research.google.com/github/Moquiuti/Arquiteturas-RAG-com-LLMs-embeddings-busca-sem-ntica-e-cria-o-de-agentes-com-LangChain/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-openai langchain-community faiss-cpu pypdf tiktoken

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
from google.colab import files

arquivos_enviados = files.upload()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

documentos = []

for nome_arquivo in arquivos_enviados.keys():
    if nome_arquivo.lower().endswith(".pdf"):
        loader = PyPDFLoader(nome_arquivo)
        paginas = loader.load()

        for pagina in paginas:
            pagina.metadata["origem"] = nome_arquivo
            pagina.metadata["categoria"] = "documento_bruto"
            pagina.metadata["tipo"] = "pdf"

        documentos.extend(paginas)

print(f"Total de páginas carregadas: {len(documentos)}")
print(documentos[0].page_content[:500])
print(documentos[0].metadata)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documentos)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["estrategia_chunking"] = "recursive_character"
    chunk.metadata["chunk_size"] = 1000
    chunk.metadata["chunk_overlap"] = 200

print(f"Total de chunks gerados: {len(chunks)}")
print(chunks[0].page_content[:500])
print(chunks[0].metadata)

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("VectorStore criada com sucesso!")

In [ ]:
!pip install -q sentence-transformers langchain-huggingface

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("VectorStore criada com sucesso!")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("VectorStore criada com embeddings open source!")

In [16]:
pergunta = "Quais são os principais pontos do documento?"

documentos_recuperados = retriever.invoke(pergunta)

for i, doc in enumerate(documentos_recuperados, start=1):
    print(f"\n--- Trecho recuperado {i} ---")
    print("Origem:", doc.metadata.get("origem"))
    print("Página:", doc.metadata.get("page"))
    print("Chunk:", doc.metadata.get("chunk_id"))
    print(doc.page_content[:1000])


--- Trecho recuperado 1 ---
Origem: Ursacol.pdf
Página: 3
Chunk: 16
biconvexo. 
Antes de usar, observe o aspecto do medicamento. Caso esteja no  prazo de validade e você observe 
alguma mudança no aspecto, consulte o farmacêutico para saber se poderá utilizá-lo. 
Todo medicamento deve ser mantido fora do alcance das crianças. 
 
6. COMO DEVO USAR ESTE MEDICAMENTO? 
Tome Ursacol
® exatamente conforme a orientação de seu médico.

--- Trecho recuperado 2 ---
Origem: Ursacol.pdf
Página: 4
Chunk: 20
refeições.  Poderá ser administrada a metade da dose diária apó s o jantar.  Ingerir os comprimidos com 
um copo de água ou leite.   
Quando o paciente se esquecer de tomar o medicamento no horário  de costume, deverá administrá-lo 
imediatamente caso não esteja muito próximo da dose subsequente. 
Siga a orientação de seu médico, respeitando sempre os horários , as doses e a duração do 
tratamento. Não interrompa o tratamento sem o conhecimento do seu médico. 
Este medicamento não deve ser part